In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os

In [3]:
!pip install -q transformers peft accelerate bitsandbytes datasets trl gradio huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.2 MB/s eta 0:00:00


In [4]:
PROJECT_PATH = '/content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill'

os.listdir(PROJECT_PATH)
!wc -l {PROJECT_PATH}/data/train.jsonl {PROJECT_PATH}/data/test.jsonl

  120 /content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill/data/train.jsonl
   30 /content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill/data/test.jsonl
  150 total


In [5]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": f"{PROJECT_PATH}/data/train_fixed.jsonl",
    "test": f"{PROJECT_PATH}/data/test_fixed.jsonl"
})

print(dataset)
print(dataset["train"][0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'metadata'],
        num_rows: 145
    })
    test: Dataset({
        features: ['input', 'output', 'metadata'],
        num_rows: 35
    })
})
{'input': 'Patient presents with chest pain radiating to the left arm and shortness of breath for the past 2 hours.', 'output': {'symptoms': ['chest pain', 'shortness of breath'], 'duration': ['2 hours', 'unspecified'], 'severity': ['severe', 'unspecified'], 'urgent': True}, 'metadata': {'clinical_domain': 'cardiac', 'generated_by': 'gpt-4o'}}


Format for Instruction Tuning


WHY WE FORMAT THE DATASET INTO A "text" FIELD
Our raw dataset has 3 separate fields:
  - "input"    → the clinical note text
  - "output"   → the correct structured JSON answer
  - "metadata" → domain, source model info
#
SFTTrainer (the fine-tuning library) needs ONE single string
 per example to feed into the model during training.

 So we combine input + output into a single "text" field
 using the Alpaca instruction format:

 ### Instruction:  → tells model what task to do
  ### Input:        → the clinical note
   ### Output:       → the correct JSON extraction

The model reads thousands of these and learns the pattern:
 "when I see a clinical note → produce structured JSON"
 This is standard instruction-tuning format used in:
Alpaca, LLaMA fine-tuning, and most clinical NLP papers.

 dataset_text_field="text" in SFTTrainer tells the trainer
which field to use for training.


In [6]:
from huggingface_hub import login
import json
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
# def format_example(example):
#     output = example['output']
#     if isinstance(output, str):
#         output = json.loads(output)

#     return {
#         "text": f"""### Instruction:
# Extract clinical information and respond with ONLY a JSON object in this exact format:
# {{"symptoms": ["symptom1", "symptom2"], "duration": ["duration1", "duration2"], "severity": ["severity1", "severity2"], "urgent": true or false}}
# Rules:
# - Match symptoms/duration/severity arrays by index (same length always)
# - Use "unspecified" if duration or severity not mentioned
# - urgent=true ONLY for: chest pain, breathing difficulty, stroke symptoms, severe bleeding, loss of consciousness
# - urgent=false for everything else including fever, back pain, headache, nausea
# - No extra text, no markdown, no code blocks, no explanation

# ### Input:
# {example['input']}

# ### Output:
# {json.dumps(output)}"""
#     }

def format_example(example):
    output = example['output']
    if isinstance(output, str):
        output = json.loads(output)

    return {
        "text": f"""<instruction>
Extract symptoms from the clinical note below. Reply with ONLY valid JSON. No comments, no markdown, no extra text.
Format: {{"symptoms": ["s1", "s2"], "duration": ["d1", "d2"], "severity": ["sev1", "sev2"], "urgent": true/false}}
Use "unspecified" if duration or severity unknown. urgent=true only for chest pain, breathing difficulty, stroke, severe bleeding.
</instruction>
<input>{example['input']}</input>
<output>{json.dumps(output)}</output>"""
    }

dataset = dataset.map(format_example)
print(dataset["train"][0]["text"])

Map:   0%|          | 0/145 [00:00<?, ? examples/s]

Map:   0%|          | 0/35 [00:00<?, ? examples/s]

<instruction>
Extract symptoms from the clinical note below. Reply with ONLY valid JSON. No comments, no markdown, no extra text.
Format: {"symptoms": ["s1", "s2"], "duration": ["d1", "d2"], "severity": ["sev1", "sev2"], "urgent": true/false}
Use "unspecified" if duration or severity unknown. urgent=true only for chest pain, breathing difficulty, stroke, severe bleeding.
</instruction>
<input>Patient presents with chest pain radiating to the left arm and shortness of breath for the past 2 hours.</input>
<output>{"symptoms": ["chest pain", "shortness of breath"], "duration": ["2 hours", "unspecified"], "severity": ["severe", "unspecified"], "urgent": true}</output>


In [8]:
# ============================================================
# LOAD MODEL + TOKENIZER
# ============================================================
# We use Gemma-3-1B — smallest Gemma that fits on free T4 GPU
# float16 reduces memory usage vs float32
# device_map="auto" automatically puts model on GPU
# ============================================================


from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "google/gemma-3-1b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype = torch.float16,
    device_map="auto"
)

print(f"Model loaded: {model_id}")
print(f"Parameters: {model.num_parameters():,}")



config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Model loaded: google/gemma-3-1b-it
Parameters: 999,885,952


Lora Configuration:

Instead of training all 1b parameters, LoRA injects tiny trainable adapters into the attention layers only

In [9]:
# r=8           → adapter rank (size of adapter, higher = more params)
# lora_alpha=16 → scaling factor, usually 2x r
# target_modules → q_proj & v_proj are the attention layers
#                  these are the most important for learning new tasks
# lora_dropout  → prevents overfitting on small dataset
# bias="none"   → don't train bias terms
# task_type     → we're doing causal language modeling
#
# Result: only ~0.15% of parameters train instead of 100%
# This is why LoRA fits on a free T4 GPU!

PEFT: parameter efficient fine-tuning: hugging face lib that contains mul techniques for fine-tuning models without training all parameters


Attention mech working:
Every time a model reads a word it asks three question:

Q (query) -> "What am i looking for?"
K (key) -> "what do i Contain?"
V (Value) -> What info do i pass forward?"

"Patient has chest pain and shortness of breath"

Q → "chest pain" asks → what else is related to me?
K → "shortness of breath" answers → I'm related, I'm a symptom too
V → passes → both should go into symptoms array

q_proj = the layer that creates Query vectors
v_proj = the layer that creates Value vectors

In [10]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()



trainable params: 1,490,944 || all params: 1,001,376,896 || trainable%: 0.1489


# Training Arguments

### num_train_epochs=3     → 3 full passes through your 120 examples
### batch_size=2           → 2 examples at a time (T4 memory limit)
### gradient_accumulation  → simulates batch of 8 (2x4) without OOM
### learning_rate=2e-4     → standard for LoRA fine-tuning
### fp16=True              → matches our float16 model
### output_dir             → saves checkpoints to your Google Drive
### report_to="none"       → disables wandb logging

In [11]:
import torch
print(torch.cuda.is_available())       # should print True
print(torch.cuda.get_device_name(0))   # should print Tesla T4

True
Tesla T4


In [12]:
!pip install -q transformers peft accelerate bitsandbytes datasets trl gradio huggingface_hub

In [13]:
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = f"{PROJECT_PATH}/models/gemma-lora"
args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=7,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    dataset_text_field="text",
    # max_seq_length=512
)
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    # tokenizer=tokenizer
    processing_class=tokenizer,

)

print("Starting training...")
trainer.train()
print("Training complete!")

Adding EOS to train dataset:   0%|          | 0/145 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/145 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Starting training...


Step,Training Loss
10,2.060156
20,1.165743
30,0.619675
40,0.386238
50,0.307075
60,0.260240
70,0.266029
80,0.267127
90,0.239010
100,0.249880


Training complete!


In [14]:
import trl
print(trl.__version__)

1.1.0


In [15]:
SAVE_PATH = f"{PROJECT_PATH}/models/gemma-lora-final"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"Model saved to: {SAVE_PATH}")

Model saved to: /content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill/models/gemma-lora-final


In [16]:
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load base model + LoRA adapter
SAVE_PATH = f"{PROJECT_PATH}/models/gemma-lora-final"

base_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-1b-it",
    torch_dtype=torch.float16,
    device_map="auto"
)
model_loaded = PeftModel.from_pretrained(base_model, SAVE_PATH)
tokenizer_loaded = AutoTokenizer.from_pretrained(SAVE_PATH)

print("Model loaded for inference!")


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Model loaded for inference!


In [17]:
def extract_clinical(text):
    prompt = f"""<instruction>
Extract symptoms from the clinical note below. Reply with ONLY valid JSON. No comments, no markdown, no extra text.
Format: {{"symptoms": ["s1", "s2"], "duration": ["d1", "d2"], "severity": ["sev1", "sev2"], "urgent": true/false}}
Use "unspecified" if duration or severity unknown. urgent=true only for chest pain, breathing difficulty, stroke, severe bleeding.
</instruction>
<input>{text}</input>
<output>"""

    inputs = tokenizer_loaded(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model_loaded.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer_loaded.eos_token_id
        )

    response = tokenizer_loaded.decode(outputs[0], skip_special_tokens=True)
    return response.split("<output>")[-1].strip()

In [18]:

test_cases = [
    "Elderly patient complaining of severe headache for 2 days, blurred vision, and confusion.",
    "Child presents with mild fever since yesterday, runny nose and occasional cough.",
    "Patient reports sharp lower back pain for a week, worse when sitting, no fever."
]

for case in test_cases:
    print(f"INPUT: {case}")
    print(f"OUTPUT: {extract_clinical(case)}")
    print("-" * 60)

INPUT: Elderly patient complaining of severe headache for 2 days, blurred vision, and confusion.
OUTPUT: {"symptoms": ["severe headache", "blurred vision", "confusion"], "duration": ["2 days", "unspecified", "unspecified"], "severity": ["severe", "unspecified", "unspecified"], "urgent": true}</output>
------------------------------------------------------------
INPUT: Child presents with mild fever since yesterday, runny nose and occasional cough.
OUTPUT: {"symptoms": ["mild fever", "runny nose", "occasional cough"], "duration": ["since yesterday", "since yesterday", "since yesterday"], "severity": ["mild", "unspecified", "unspecified"], "urgent": false}</output>
------------------------------------------------------------
INPUT: Patient reports sharp lower back pain for a week, worse when sitting, no fever.
OUTPUT: {"symptoms": ["lower back pain", "sitting pain", "back pain"], "duration": ["one week", "one week", "unspecified"], "severity": ["moderate", "moderate", "unspecified"], "ur

In [19]:
test_cases = [
    "I've been feeling off for a few days, my chest feels weird and I get tired just walking to the kitchen.",
    "Kid woke up hot last night, been sneezing a lot, seems fine otherwise but a little cranky.",
    "My back's been killing me since last week, not sure if I pulled something, hurts more when I sit too long.",
    "Been having these headaches on and off, nothing crazy but they keep coming back, sometimes feel dizzy too.",
    "Stomach's been bothering me since yesterday, went to the bathroom like 4 times, feeling pretty drained.",
]

for case in test_cases:
    print(f"INPUT: {case}")
    print(f"OUTPUT: {extract_clinical(case)}")
    print("-" * 60)

INPUT: I've been feeling off for a few days, my chest feels weird and I get tired just walking to the kitchen.
OUTPUT: {"symptoms": ["chest pain", "fatigue"], "duration": ["a few days", "a few days"], "severity": ["unspecified", "unspecified"], "urgent": false}</output>
------------------------------------------------------------
INPUT: Kid woke up hot last night, been sneezing a lot, seems fine otherwise but a little cranky.
OUTPUT: {"symptoms": ["fever", "sneezing", "irritability"], "duration": ["unspecified", "unspecified", "unspecified"], "severity": ["mild", "unspecified", "unspecified"], "urgent": false}</output>
------------------------------------------------------------
INPUT: My back's been killing me since last week, not sure if I pulled something, hurts more when I sit too long.
OUTPUT: {"symptoms": ["back pain", "back pain"], "duration": ["since last week", "since last week"], "severity": ["unspecified", "unspecified"], "urgent": false}</output>
---------------------------

In [20]:
# ============================================================
# EVALUATION — Fixed for inconsistent symptom formats
# ============================================================

import json

def parse_output(text):
    """Safely parse model output to dict"""
    try:
        text = text.strip()
        if text.startswith("```"):
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        return json.loads(text.strip()), True
    except:
        return {}, False

def flatten_symptoms(symptoms):
    """Handle both string and dict symptom formats"""
    flat = []
    for s in symptoms:
        if isinstance(s, str):
            flat.append(s.lower())
        elif isinstance(s, dict):
            # nested format — extract symptom name
            name = s.get("symptom") or s.get("name") or s.get("description") or ""
            if name:
                flat.append(name.lower())
    return flat

def symptom_f1(pred_symptoms, true_symptoms):
    pred_set = set(flatten_symptoms(pred_symptoms))
    true_set = set(flatten_symptoms(true_symptoms))

    tp = len(pred_set & true_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)

    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall    = tp / (tp + fn) if tp + fn > 0 else 0
    f1        = 2 * (precision * recall) / (precision + recall) if precision + recall > 0 else 0
    return round(f1, 3)

def urgent_accuracy(pred, true):
    return pred.get("urgent") == true.get("urgent")

# Run on test set
valid_json_count = 0
f1_scores = []
urgent_correct = 0
results = []

for example in dataset["test"]:
    raw_output = extract_clinical(example["input"])
    pred, is_valid = parse_output(raw_output)
    true = example["output"]

    f1 = 0
    if is_valid:
        valid_json_count += 1
        f1 = symptom_f1(pred.get("symptoms", []), true.get("symptoms", []))
        f1_scores.append(f1)
        if urgent_accuracy(pred, true):
            urgent_correct += 1

    results.append({
        "input": example["input"],
        "expected": true,
        "predicted": pred,
        "valid_json": is_valid,
        "f1": f1
    })

# Print summary
total = len(dataset["test"])
avg_f1 = sum(f1_scores)/len(f1_scores) if f1_scores else 0

print("=" * 50)
print("EVALUATION RESULTS — Gemma-3-1B LoRA")
print("=" * 50)
print(f"Total test examples : {total}")
print(f"Valid JSON rate     : {valid_json_count}/{total} ({100*valid_json_count/total:.1f}%)")
print(f"Avg Symptom F1      : {avg_f1:.3f}")
print(f"Urgent Accuracy     : {urgent_correct}/{valid_json_count} ({100*urgent_correct/max(valid_json_count,1):.1f}%)")
print("=" * 50)

EVALUATION RESULTS — Gemma-3-1B LoRA
Total test examples : 35
Valid JSON rate     : 0/35 (0.0%)
Avg Symptom F1      : 0.000
Urgent Accuracy     : 0/0 (0.0%)


In [23]:
# ============================================================
# EVALUATION
# ============================================================

import json

def parse_output(text):
    """Safely parse model output to dict"""
    try:
        text = text.strip()
        # Model was trained to close the <output> tag — strip it
        text = text.replace("</output>", "").strip()
        if text.startswith("```"):
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        return json.loads(text.strip()), True
    except:
        return {}, False

def flatten_symptoms(symptoms):
    flat = []
    for s in symptoms:
        if isinstance(s, str):
            flat.append(s.lower())
        elif isinstance(s, dict):
            name = s.get("symptom") or s.get("name") or s.get("description") or ""
            if name:
                flat.append(name.lower())
    return flat

def symptom_f1(pred_symptoms, true_symptoms):
    pred_set = set(flatten_symptoms(pred_symptoms))
    true_set = set(flatten_symptoms(true_symptoms))

    tp = len(pred_set & true_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)

    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall    = tp / (tp + fn) if tp + fn > 0 else 0
    f1        = 2 * (precision * recall) / (precision + recall) if precision + recall > 0 else 0
    return round(f1, 3)

def urgent_accuracy(pred, true):
    return pred.get("urgent") == true.get("urgent")

# Run on test set
valid_json_count = 0
f1_scores = []
urgent_correct = 0
results = []

for example in dataset["test"]:
    raw_output = extract_clinical(example["input"])
    pred, is_valid = parse_output(raw_output)
    true = example["output"]

    f1 = 0
    if is_valid:
        valid_json_count += 1
        f1 = symptom_f1(pred.get("symptoms", []), true.get("symptoms", []))
        f1_scores.append(f1)
        if urgent_accuracy(pred, true):
            urgent_correct += 1

    results.append({
        "input": example["input"],
        "expected": true,
        "predicted": pred,
        "valid_json": is_valid,
        "f1": f1
    })

# Print summary
total = len(dataset["test"])
avg_f1 = sum(f1_scores)/len(f1_scores) if f1_scores else 0

print("=" * 50)
print("EVALUATION RESULTS — Gemma-3-1B LoRA")
print("=" * 50)
print(f"Total test examples : {total}")
print(f"Valid JSON rate     : {valid_json_count}/{total} ({100*valid_json_count/total:.1f}%)")
print(f"Avg Symptom F1      : {avg_f1:.3f}")
print(f"Urgent Accuracy     : {urgent_correct}/{valid_json_count} ({100*urgent_correct/max(valid_json_count,1):.1f}%)")
print("=" * 50)

EVALUATION RESULTS — Gemma-3-1B LoRA
Total test examples : 35
Valid JSON rate     : 35/35 (100.0%)
Avg Symptom F1      : 0.781
Urgent Accuracy     : 30/35 (85.7%)
